In [ ]:
import pandas as pd
import neuralset as ns
from neuralset.features import HuggingFaceText
import numpy as np
import string
import torch

df = pd.read_csv("datasets/svo_word_level.csv")

In [ ]:
def make_sentence(words):
    s = ""
    cum_s = []
    indices = []
    for word in words:
        if word in string.punctuation:
            s = s.strip()
        indices.append(len(s))
        s += word
        cum_s.append(s)
        s += " "

    return (s.strip(), indices, cum_s)


df["sentence"] = df.groupby("sentence_id").word.transform(
    lambda words: make_sentence(words)[0]
)
df["sentence_char"] = df.groupby("sentence_id").word.transform(
    lambda words: make_sentence(words)[1]
)
df["context"] = df.groupby("sentence_id").word.transform(
    lambda words: make_sentence(words)[2]
)

In [ ]:
# For each event, we need to specify how these discrete events
# can be converted into a dense time series.
feature = ns.features.SpacyEmbedding(language="english", aggregation="sum")
data = feature(events, start=10.0, duration=3.0)
(n_dims,) = data.shape

# We may want to get a dynamic event
feature = ns.features.SpacyEmbedding(
    frequency=100.0, language="english", aggregation="sum"
)
data = feature(events, start=10.0, duration=3.0)
n_dims, n_times = data.shape

# To make a DataLoader we need to generate the segments (data time windows)
# For this, we can use the `ns` accessor
segments = ns.segments.list_segments(
    events, idx=events.type == "Word", start=-0.3, duration=2.0
)
segment1, segment2 = segments

# We then need to define the dataset (segments + features) and run it through the dataloader
ds = ns.SegmentDataset(features={"embedding": feature}, segments=segments)
# for batch in DataLoader(ds, collate_fn=ds.collate_fn, batch_size=2):
#     break
# batch_size, n_dims, n_times = batch.data['embedding'].shape